# Gemma-2-2B: fact preparation (template choice, candidate facts, keep facts answered correctly in EN, ES and HI)
Inputs: none (the code is fetched from GitHub; the same `babel-bundle` dataset works as a fallback). If the run stops at the time limit, run it again with its own output attached and it resumes. Output: `babel_new_results.zip`.

In [ ]:
import io, os, shutil, subprocess, sys, time, urllib.request, zipfile
STEP = "babel_gemma2"
HOURS = 10.5  # jobs stop cleanly after this, then everything is packaged
REPO, REF = "praann07/babel-effect", "main"  # REF may be a commit id to pin the code
SRC = os.environ.get("BABEL_SRC_DIR", "/tmp/babel_src")
INPUT = os.environ.get("BABEL_INPUT", "/kaggle/input")


def from_github():
    url = os.environ.get("BABEL_ZIP_URL") or "https://codeload.github.com/" + REPO + "/zip/" + REF
    err = ""
    for i in range(4):
        try:
            z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url, timeout=90).read()))
            for d in (SRC, SRC + "_x"):
                shutil.rmtree(d, ignore_errors=True)
            z.extractall(SRC + "_x")
            shutil.move(os.path.join(SRC + "_x", os.listdir(SRC + "_x")[0]), SRC)
            return z.comment.decode() or REF  # GitHub stores the commit id in the zip comment
        except Exception as e:
            err = type(e).__name__ + ": " + str(e)[:150]
            time.sleep(5 * (i + 1))
    print("GitHub download failed (" + err + "); looking for the babel-bundle dataset instead.")
    return None


def from_dataset():
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        if {"driver.py", "steps.py", "dtype_check.py"} <= set(files):
            return os.path.join(root, "driver.py")
    for root, dirs, files in os.walk(INPUT, followlinks=True):
        if "babel_bundle.zip" in files:
            zipfile.ZipFile(os.path.join(root, "babel_bundle.zip")).extractall("/tmp/babel_zip")
            return "/tmp/babel_zip/kaggle/driver.py"
    return None


env = dict(os.environ, BABEL_STEP=STEP, HOURS=str(HOURS), PYTHONUNBUFFERED="1")
sha = from_github()
if sha:
    drv = os.path.join(SRC, "kaggle", "driver.py")
    env.update(BABEL_SRC=SRC, BABEL_SRC_SHA=sha)
    print("code from GitHub " + REPO + " @ " + sha)
else:
    drv = from_dataset()
if not drv:
    print("No code found: GitHub was unreachable (is Internet on for this notebook?) and no babel-bundle dataset is attached.")
else:
    try:
        from kaggle_secrets import UserSecretsClient
        env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("HF_TOKEN secret not readable here (" + type(e).__name__ + ": " + str(e)[:150] + "); the driver will try again itself.")
    p = subprocess.Popen([sys.executable, drv], env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    print("driver exit code:", p.returncode, "(0 = everything finished)")

In [ ]:
import json, os
f = "/kaggle/working/results/RUN_SUMMARY.json"
print(json.dumps(json.load(open(f)), indent=1) if os.path.exists(f) else "no summary (see the messages above)")